# Explore Benchmark Results

Load results from a completed benchmark run and explore individual variants.
Point `BENCHMARK_DIR` at a benchmark output directory.

In [ ]:
from jax import config
config.update('jax_enable_x64', True)

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from rkpcn_analysis.benchmark import (
    load_benchmark_results, load_benchmark_meta,
    print_benchmark_summary, compute_benchmark_w2,
    plot_benchmark_comparison,
)
from rkpcn_analysis.reconstruct import load_saved_data
from rkpcn_analysis.diagnostics import autocorrelation, integrated_autocorrelation_time
from rkpcn_analysis.plots import plot_density_heatmaps
from uncprop.utils.grid import Grid
from uncprop.utils.diagnostics import compute_ess

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['font.size'] = 12

## 1. Configuration

In [ ]:
# --- Point these at your benchmark and experiment data ---
EXPERIMENT_NAME = 'vsem'
SETUP = 'clip_gp_N4'
REP = 0

repo_root = Path('.').resolve().parents[1]
BENCHMARK_DIR = repo_root / 'out' / EXPERIMENT_NAME / 'benchmarks' / f'{SETUP}_rep{REP}'
EXPERIMENT_DIR = repo_root / 'out' / EXPERIMENT_NAME

print(f'Benchmark dir: {BENCHMARK_DIR}')
print(f'Experiment dir: {EXPERIMENT_DIR}')

## 2. Load results

In [ ]:
results = load_benchmark_results(BENCHMARK_DIR)
meta = load_benchmark_meta(BENCHMARK_DIR)
saved = load_saved_data(EXPERIMENT_DIR, SETUP, REP)

print(f'Variants: {list(results.keys())}')
print(f'Has traces: {any(d["trace"] is not None for d in results.values())}')

# Load grid for density plots
grid = None
if saved['grid_info'] is not None:
    gi = saved['grid_info']
    grid = Grid(
        low=np.array(gi['low']),
        high=np.array(gi['high']),
        n_points_per_dim=np.array(gi['n_points_per_dim'], dtype=int),
        dim_names=list(gi['dim_names']) if 'dim_names' in gi else None,
    )
    par_names = list(gi['dim_names']) if 'dim_names' in gi else None
else:
    par_names = None
    print('Warning: no grid_info.npz — density plots unavailable')

gd = saved.get('grid_densities')

## 3. Summary table

In [ ]:
print_benchmark_summary(results, par_names=par_names)

## 4. Density heatmaps (exact, mean, eup, ep)

In [ ]:
if gd is not None and grid is not None:
    fig, _ = plot_density_heatmaps(gd, grid, names=['exact', 'mean', 'eup', 'ep'])
    plt.show()

## 5. Scatter plots vs EP

In [ ]:
reference_samples = {}
if saved['samples'] is not None:
    for name in ['exact', 'mean', 'eup']:
        if name in saved['samples']:
            reference_samples[name] = np.array(saved['samples'][name])

design_points = None
if saved['setup_info'] is not None and 'design_x' in saved['setup_info']:
    design_points = np.array(saved['setup_info']['design_x'])

if gd is not None and grid is not None:
    fig, _ = plot_benchmark_comparison(
        results, gd['ep'], grid,
        reference_samples=reference_samples,
        design_points=design_points, thin=3)
    plt.show()

## 6. W2 to EP

In [ ]:
if gd is not None and grid is not None:
    w2 = compute_benchmark_w2(
        results, gd['ep'], grid,
        reference_samples=reference_samples, thin=3)

---

## 7. Detailed variant diagnostics

Select a variant below and get detailed MCMC diagnostics.

In [ ]:
# --- Select variant to inspect ---
VARIANT = 'rho99'  # change to any label from the benchmark

data = results[VARIANT]
s = data['summary']

print(f'=== Variant: {VARIANT} ===')
print(f'  rho: {s["rho"]}')
print(f'  n_u_steps: {s.get("n_u_steps", 1)}')
print(f'  n_burnin: {s["n_burnin"]}')
print(f'  n_post_burnin: {s["n_post_burnin"]}')
print(f'  accept_rate: {s["accept_rate"]:.4f}')
print(f'  min_ESS: {s["min_ess"]:.1f}')
print(f'  ESS: {s["ess"]}')
print(f'  IAT(logdensity): {s["iat_logdensity"]:.1f}')
print(f'  runtime: {s.get("runtime", 0):.1f}s')
print(f'  prop_cov_diag: {s.get("prop_cov_diag", "N/A")}')

samp = data['post_burnin']
if samp is not None:
    print(f'\n  Sample stats:')
    for j, p in enumerate(par_names or [f'u{j}' for j in range(samp.shape[1])]):
        print(f'    {p}: mean={samp[:, j].mean():.4f}, std={samp[:, j].std():.4f}, '
              f'range=[{samp[:, j].min():.4f}, {samp[:, j].max():.4f}]')

In [ ]:
# Trace plots (requires --save-trace during benchmark run)
trace = data.get('trace')

if trace is not None:
    n_burnin = s['n_burnin']
    positions = trace['positions']
    logdens = trace['logdensities']
    accept_probs = trace['accept_probs']
    d = positions.shape[1]
    names = par_names or [f'u{j}' for j in range(d)]

    # Log-density trace
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(logdens, linewidth=0.3, alpha=0.7)
    ax.axvline(n_burnin, color='red', linestyle='--', linewidth=1, label='burnin end')
    ax.set_title(f'{VARIANT}: log-density trace')
    ax.set_xlabel('Iteration')
    ax.legend()
    plt.show()

    # Parameter traces (post-burnin)
    fig, axes = plt.subplots(1, d, figsize=(7*d, 3))
    if d == 1:
        axes = [axes]
    for j, ax in enumerate(axes):
        ax.plot(positions[n_burnin:, j], linewidth=0.3, alpha=0.7)
        ax.set_title(f'{names[j]} (post-burnin)')
        ax.set_xlabel('Iteration')
    fig.suptitle(VARIANT, fontsize=13, y=1.02)
    fig.tight_layout()
    plt.show()

    # Rolling acceptance rate
    window = 500
    rolling = np.convolve(accept_probs, np.ones(window)/window, mode='valid')
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(rolling, linewidth=0.5)
    ax.axvline(n_burnin, color='red', linestyle='--', linewidth=1)
    ax.set_title(f'{VARIANT}: rolling accept rate (window={window})')
    ax.set_xlabel('Iteration')
    plt.show()

    # ACF
    max_lag = min(3000, positions.shape[0] - n_burnin)
    fig, axes = plt.subplots(1, d+1, figsize=(5*(d+1), 4))
    for j in range(d):
        acf = autocorrelation(positions[n_burnin:, j], max_lag=max_lag)
        axes[j].plot(acf)
        axes[j].axhline(0, color='black', linewidth=0.5)
        axes[j].set_title(f'ACF: {names[j]}')
        axes[j].set_xlabel('Lag')
    acf_ld = autocorrelation(logdens[n_burnin:], max_lag=max_lag)
    axes[d].plot(acf_ld)
    axes[d].axhline(0, color='black', linewidth=0.5)
    axes[d].set_title('ACF: log-density')
    axes[d].set_xlabel('Lag')
    fig.suptitle(VARIANT, fontsize=13, y=1.02)
    fig.tight_layout()
    plt.show()

    # Recompute ESS from full trace
    ess_full = compute_ess(positions[n_burnin:])
    print(f'ESS from full trace: {ess_full}')
else:
    print('No trace data — run benchmark with --save-trace to enable trace plots.')
    print('Summary stats from saved diagnostics are shown above.')

In [ ]:
# 2D scatter of this variant vs EP
if samp is not None and gd is not None and grid is not None:
    from rkpcn_analysis.plots import plot_samples_vs_ep

    single_result = {VARIANT: {'post_burnin': samp, 'label': VARIANT}}
    fig, _ = plot_samples_vs_ep(
        single_result, ep_density=gd['ep'], grid=grid,
        reference_samples=reference_samples, thin=3)
    plt.show()

## 8. Compare two variants side-by-side

In [ ]:
VAR_A = 'cut'
VAR_B = 'rho99'

if gd is not None and grid is not None:
    subset = {k: results[k] for k in [VAR_A, VAR_B] if k in results}
    fig, _ = plot_benchmark_comparison(
        subset, gd['ep'], grid,
        reference_samples=reference_samples,
        design_points=design_points, thin=3)
    plt.show()

for v in [VAR_A, VAR_B]:
    if v in results:
        s = results[v]['summary']
        print(f'{v}: accept={s["accept_rate"]:.4f}, '
              f'min_ESS={s["min_ess"]:.1f}, '
              f'IAT(ld)={s["iat_logdensity"]:.1f}')